
# SeaFour Retrieval Engine Competition Notebook — Maksym-Style Revised Edition

**Team:** SeaFour  
**Members:** Mehdi (Soroosh) Aghaei, Maksym DOLHOV, Khánh Bảo

This notebook is intentionally built on the strengths of the cleaner Maksym-style structure: a compact, runnable, cache-aware pipeline with explicit evaluation, model comparison, final submission generation, and a written report at the end.



## What this notebook does

This notebook implements the two competition phases in one reproducible workflow.

For retrieval, it compares:
- **TF-IDF**
- **BM25+**
- **Embedding-based retrieval**
- **Hybrid cooperation strategies** such as reciprocal-rank fusion and category-aware reranking

For Phase 2, it also trains a **query category classifier** and uses the predicted domain to improve reranking.

The notebook is designed to answer four practical questions:
1. Which retrieval family performs best offline on held-out validation?
2. What value of `top_k` is best for the weighted leaderboard objective?
3. Does classifier-guided reranking actually help?
4. Which exact configuration should be used for the final Kaggle submission?


In [ ]:

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple
import csv
import hashlib
import json
import os
import pickle
import re
import warnings

import numpy as np
import pandas as pd

from IPython.display import Markdown, display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split

pd.set_option("display.max_colwidth", 140)
pd.set_option("display.precision", 5)
warnings.filterwarnings("ignore")


In [ ]:

@dataclass(frozen=True)
class Config:
    phase: int = 2  # Set to 1 for Phase 1 submission.
    random_state: int = 42
    validation_size: float = 0.20

    output_filename: str = "solutions_SeaFour.csv"
    report_team_name: str = "SeaFour"

    document_text_columns: Tuple[str, ...] = ("title", "text", "tags")
    retrieval_query_columns: Tuple[str, ...] = ("title", "text", "tags")
    classifier_query_columns: Tuple[str, ...] = ("title", "text", "tags")
    weight_columns: Tuple[str, ...] = ("title", "tags")

    use_stopwords: bool = True
    use_stemming: bool = False
    token_pattern: str = r"[a-z0-9]+"

    tfidf_ngram_range: Tuple[int, int] = (1, 2)
    tfidf_min_df: int = 2
    tfidf_sublinear_tf: bool = True

    bm25_k1: float = 1.5
    bm25_b: float = 0.75
    bm25_delta: float = 1.0

    embedding_model_names: Tuple[str, ...] = (
        "sentence-transformers/all-MiniLM-L6-v2",
        "sentence-transformers/multi-qa-MiniLM-L6-cos-v1",
    )
    embedding_batch_size: int = 128
    embedding_candidate_multiplier: int = 4

    top_k_grid: Tuple[int, ...] = (20, 50, 100, 200, 500, 1000)
    rrf_k: int = 60
    category_bonus: float = 0.05

    enable_disk_cache: bool = True
    enable_category_bonus_experiments: bool = True

CFG = Config()
print(CFG)



## Runtime and data-path resolution

The notebook supports three common execution environments:
- **Kaggle** competition runtime
- **Colab** with files mounted from Drive
- **Local** execution near a `data/` directory

It expects the standard competition files:
- `docs.json`
- `queries_train.json`
- `queries_test.json`
- `qgts_train.json`
- `submission.csv`


In [ ]:

def detect_runtime_environment() -> str:
    if Path("/kaggle/input").exists():
        return "kaggle"
    try:
        import google.colab  # type: ignore  # noqa: F401
        return "colab"
    except Exception:
        return "local"


def find_data_dir() -> Path:
    runtime_env = detect_runtime_environment()

    if runtime_env == "kaggle":
        for root, _, files in os.walk("/kaggle/input"):
            filenames = set(files)
            if {"docs.json", "queries_train.json", "queries_test.json", "qgts_train.json", "submission.csv"}.issubset(filenames):
                return Path(root)

    candidates = [
        Path.cwd() / "data",
        Path.cwd().parent / "data",
        Path.cwd(),
        Path("/mnt/data"),
    ]
    for candidate in candidates:
        if (candidate / "queries_train.json").exists() and (candidate / "queries_test.json").exists():
            return candidate

    raise FileNotFoundError(
        "Could not locate the competition data directory. Place the JSON files in a local `data/` folder or run in Kaggle."
    )


def ensure_dir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path


DATA_DIR = find_data_dir()
WORK_DIR = Path.cwd()
CACHE_DIR = ensure_dir(WORK_DIR / "cache_retrieval_engine")
OUTPUT_PATH = WORK_DIR / CFG.output_filename

print("Runtime:", detect_runtime_environment())
print("Data dir:", DATA_DIR.resolve())
print("Cache dir:", CACHE_DIR.resolve())
print("Output:", OUTPUT_PATH.resolve())


In [ ]:

def load_json_frame(path: Path, name: str) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"{name} not found: {path}")
    return pd.read_json(path)


def require_columns(frame: pd.DataFrame, required: Sequence[str], name: str) -> None:
    missing = [c for c in required if c not in frame.columns]
    if missing:
        raise ValueError(f"{name} is missing required columns: {missing}")


def ensure_unique_ids(frame: pd.DataFrame, name: str) -> None:
    require_columns(frame, ["id"], name)
    if not frame["id"].astype(str).is_unique:
        raise ValueError(f"{name} contains duplicate ids.")


docs_raw_df = load_json_frame(DATA_DIR / "docs.json", "docs.json")
queries_train_raw_df = load_json_frame(DATA_DIR / "queries_train.json", "queries_train.json")
queries_test_raw_df = load_json_frame(DATA_DIR / "queries_test.json", "queries_test.json")
qgts_path = DATA_DIR / "qgts_train.json"
sample_submission_path = DATA_DIR / "submission.csv"

require_columns(docs_raw_df, ["id", "title", "text", "tags"], "docs")
require_columns(queries_train_raw_df, ["id", "title", "text", "tags"], "queries_train")
require_columns(queries_test_raw_df, ["id", "title", "text", "tags"], "queries_test")
ensure_unique_ids(docs_raw_df, "docs")
ensure_unique_ids(queries_train_raw_df, "queries_train")
ensure_unique_ids(queries_test_raw_df, "queries_test")

print(f"Documents      : {len(docs_raw_df):,}")
print(f"Train queries  : {len(queries_train_raw_df):,}")
print(f"Test queries   : {len(queries_test_raw_df):,}")
display(docs_raw_df.head(2))
display(queries_train_raw_df.head(2))



## Text construction and normalization

The corpus contains multiple useful fields: title, text, tags, and often category for documents.  
We build a single `content` field for retrieval while still keeping structured columns for later experiments.

The preprocessing is intentionally moderate:
- lowercase normalization
- separator cleanup (`-`, `_`, `/`)
- regex tokenization
- optional stopword removal
- optional stemming

This is usually safer than heavy cleaning because dense retrieval and BM25+ both benefit from keeping important lexical signals intact.


In [ ]:

TOKEN_RE = re.compile(CFG.token_pattern)
_NLTK_READY = False
_NLTK_STOPWORDS = None
_NLTK_STEMMER = None


def _init_nltk() -> bool:
    global _NLTK_READY, _NLTK_STOPWORDS, _NLTK_STEMMER
    if _NLTK_READY:
        return True
    try:
        import nltk
        from nltk.corpus import stopwords
        from nltk.stem import PorterStemmer

        try:
            _ = stopwords.words("english")
        except LookupError:
            nltk.download("stopwords", quiet=True)

        _NLTK_STOPWORDS = set(stopwords.words("english"))
        _NLTK_STEMMER = PorterStemmer()
        _NLTK_READY = True
        return True
    except Exception:
        return False


def value_to_text(value: Any) -> str:
    if value is None:
        return ""
    if isinstance(value, (list, tuple)):
        return " ".join(str(v) for v in value)
    if pd.isna(value):
        return ""
    return str(value)


def normalize_text(text: Any) -> str:
    text = value_to_text(text).lower()
    text = re.sub(r"[-_/]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def tokenize(text: Any) -> List[str]:
    tokens = TOKEN_RE.findall(normalize_text(text))

    if not (CFG.use_stopwords or CFG.use_stemming):
        return tokens
    if not _init_nltk():
        return tokens

    if CFG.use_stopwords:
        tokens = [t for t in tokens if t not in _NLTK_STOPWORDS]
    if CFG.use_stemming:
        tokens = [_NLTK_STEMMER.stem(t) for t in tokens]
    return tokens


def build_content_column(frame: pd.DataFrame, columns: Sequence[str], weight_columns: Sequence[str] = ()) -> pd.DataFrame:
    out = frame.copy()
    weight_columns = set(weight_columns)
    for col in columns:
        if col not in out.columns:
            out[col] = ""

    merged = []
    for _, row in out[columns].iterrows():
        pieces = []
        for col in columns:
            text = normalize_text(row[col])
            pieces.append(text)
            if col in weight_columns and text:
                pieces.append(text)
        merged.append(" ".join(pieces).strip())

    out["id"] = out["id"].astype(str)
    out["content"] = merged
    out["content_no_tags"] = out.apply(
        lambda row: " ".join(normalize_text(row[col]) for col in columns if col != "tags").strip(),
        axis=1,
    )
    return out


docs_df = build_content_column(docs_raw_df, CFG.document_text_columns, CFG.weight_columns)
queries_train_df = build_content_column(queries_train_raw_df, CFG.classifier_query_columns, CFG.weight_columns)
queries_test_df = build_content_column(queries_test_raw_df, CFG.classifier_query_columns, CFG.weight_columns)

for frame in (docs_df, queries_train_df, queries_test_df):
    if "category" in frame.columns:
        frame["category"] = frame["category"].astype(str)

print(docs_df[["id", "content"]].head(2).to_string(index=False))



## Ground truth, held-out validation, and offline metrics

A notebook can look strong while still being misleading if it evaluates on the same queries it used to tune the system.  
This revision avoids that problem by creating a **stratified train/validation split** using the query category labels.

Offline evaluation uses the same weighted competition logic:

\[
	ext{score} = 0.25\cdot	ext{Recall} + 0.25\cdot	ext{Precision} + 0.25\cdot	ext{MRR} + 0.25\cdot	ext{Accuracy}
\]

For Phase 1, accuracy is forced to 0. For Phase 2, the classifier becomes active.


In [ ]:

with open(qgts_path, "r", encoding="utf-8") as f:
    qgts_raw = json.load(f)


def build_gold_frame(qgts: Dict[str, Any]) -> pd.DataFrame:
    rows = []
    for query_id, info in qgts.items():
        rows.append({
            "id": str(query_id),
            "gold_category": info.get("category"),
            "gold_total_relevant": int(info.get("total_relevant_docs", len(info.get("relevant_doc_ids", [])))),
            "gold_relevant_doc_ids": {str(item["doc_id"]) for item in info.get("relevant_doc_ids", [])},
        })
    return pd.DataFrame(rows)


gold_df = build_gold_frame(qgts_raw)
queries_train_df = queries_train_df.merge(gold_df, on="id", how="inner")

train_fold, val_fold = train_test_split(
    queries_train_df,
    test_size=CFG.validation_size,
    random_state=CFG.random_state,
    stratify=queries_train_df["gold_category"],
)
train_fold = train_fold.reset_index(drop=True)
val_fold = val_fold.reset_index(drop=True)

print("Train fold:", train_fold.shape)
print("Validation fold:", val_fold.shape)
display(train_fold[["id", "gold_category", "gold_total_relevant"]].head())


def evaluate_retrieval(pred_map: Dict[str, List[str]], eval_df: pd.DataFrame, k: int) -> Dict[str, float]:
    recalls, precisions, mrrs = [], [], []
    for _, row in eval_df.iterrows():
        qid = str(row["id"])
        predicted = [str(x) for x in pred_map.get(qid, [])[:k]]
        relevant = set(row["gold_relevant_doc_ids"])
        total_relevant = int(row["gold_total_relevant"])

        hits = sum(doc_id in relevant for doc_id in predicted)
        recalls.append(hits / total_relevant if total_relevant > 0 else 0.0)
        precisions.append(hits / len(predicted) if predicted else 0.0)

        rr = 0.0
        for rank, doc_id in enumerate(predicted, start=1):
            if doc_id in relevant:
                rr = 1.0 / rank
                break
        mrrs.append(rr)

    return {
        f"recall@{k}": float(np.mean(recalls)) if recalls else 0.0,
        f"precision@{k}": float(np.mean(precisions)) if precisions else 0.0,
        f"mrr@{k}": float(np.mean(mrrs)) if mrrs else 0.0,
    }


def evaluate_classification(y_true: Sequence[str], y_pred: Sequence[str]) -> Dict[str, float]:
    return {"accuracy": float(accuracy_score(y_true, y_pred))}


def combined_score(metrics: Dict[str, float], k: int, phase: int) -> float:
    recall = metrics.get(f"recall@{k}", 0.0)
    precision = metrics.get(f"precision@{k}", 0.0)
    mrr = metrics.get(f"mrr@{k}", 0.0)
    acc = metrics.get("accuracy", 0.0) if phase == 2 else 0.0
    return 0.25 * recall + 0.25 * precision + 0.25 * mrr + 0.25 * acc



## Query category classifier

The classifier is intentionally simple and strong:
- a **tag-to-category prior** for very direct signals
- a **TF-IDF + Logistic Regression** text model as the main classifier
- a fallback rule: if tags strongly indicate a category, use that; otherwise use the text classifier

This design is fast, interpretable, and easy to debug.


In [ ]:

from collections import Counter, defaultdict


def build_tag_category_map(train_df: pd.DataFrame) -> Dict[str, Counter]:
    tag_to_cat: Dict[str, Counter] = defaultdict(Counter)
    for _, row in train_df.iterrows():
        category = str(row["gold_category"])
        for tag in row.get("tags", []) or []:
            tag_to_cat[str(tag)][category] += 1
    return tag_to_cat


class QueryCategoryClassifier:
    def __init__(self) -> None:
        self.tag_to_cat = None
        self.vectorizer = TfidfVectorizer(
            lowercase=False,
            tokenizer=tokenize,
            ngram_range=CFG.tfidf_ngram_range,
            min_df=1,
            sublinear_tf=True,
        )
        self.model = LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=CFG.random_state,
        )
        self.is_fit = False

    def fit(self, train_df: pd.DataFrame) -> "QueryCategoryClassifier":
        self.tag_to_cat = build_tag_category_map(train_df)
        X = self.vectorizer.fit_transform(train_df["content_no_tags"])
        y = train_df["gold_category"].astype(str)
        self.model.fit(X, y)
        self.is_fit = True
        return self

    def predict_from_tags(self, tags: Iterable[str]) -> Optional[str]:
        counts = Counter()
        for tag in tags or []:
            tag = str(tag)
            if self.tag_to_cat is not None and tag in self.tag_to_cat:
                counts.update(self.tag_to_cat[tag])
        if not counts:
            return None
        return counts.most_common(1)[0][0]

    def predict(self, df: pd.DataFrame) -> List[str]:
        assert self.is_fit, "Fit the classifier first."
        X = self.vectorizer.transform(df["content_no_tags"])
        text_preds = self.model.predict(X)
        out = []
        for i, (_, row) in enumerate(df.iterrows()):
            tag_pred = self.predict_from_tags(row.get("tags", []))
            out.append(tag_pred if tag_pred is not None else str(text_preds[i]))
        return out


clf_model = QueryCategoryClassifier().fit(train_fold)
val_category_preds = clf_model.predict(val_fold)
clf_metrics = evaluate_classification(val_fold["gold_category"].astype(str).tolist(), val_category_preds)
print(clf_metrics)
print(classification_report(val_fold["gold_category"].astype(str), val_category_preds))



## Cache helpers

Dense retrieval is usually the expensive part.  
To keep the notebook practical, document embeddings and indices are cached on disk. This makes repeated experiments much cheaper.


In [ ]:

OBJECT_CACHE: Dict[str, Any] = {}
ARRAY_CACHE: Dict[str, np.ndarray] = {}


def safe_name(value: Any) -> str:
    return re.sub(r"[^A-Za-z0-9._-]+", "_", str(value))


def hash_payload(payload: Dict[str, Any]) -> str:
    raw = json.dumps(payload, sort_keys=True, default=str).encode("utf-8")
    return hashlib.sha1(raw).hexdigest()[:16]


def dataframe_fingerprint(df: pd.DataFrame, columns: Sequence[str]) -> str:
    hasher = hashlib.sha1()
    hasher.update(str(len(df)).encode("utf-8"))
    for col in columns:
        hasher.update(col.encode("utf-8"))
        values = pd.util.hash_pandas_object(df[col].astype(str), index=False).values
        hasher.update(values.tobytes())
    return hasher.hexdigest()[:16]


def load_pickle(path: Path) -> Any:
    with open(path, "rb") as f:
        return pickle.load(f)


def save_pickle(path: Path, obj: Any) -> None:
    with open(path, "wb") as f:
        pickle.dump(obj, f, protocol=pickle.HIGHEST_PROTOCOL)



## Retrieval models

This notebook compares three core retrieval families required by the competition:

1. **TF-IDF** — a sparse lexical baseline using cosine similarity  
2. **BM25+** — a stronger lexical baseline with document-length normalization  
3. **Dense retrieval** — embedding-based semantic search using Sentence-Transformers

On top of that, it also tests **cooperation strategies**:
- **RRF fusion** between sparse and dense rankings
- **Category-aware reranking** using the Phase 2 classifier


In [ ]:

try:
    from rank_bm25 import BM25Plus
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "rank_bm25"])
    from rank_bm25 import BM25Plus


def top_k_from_scores(scores: np.ndarray, top_k: int) -> np.ndarray:
    top_k = min(top_k, len(scores))
    if top_k == len(scores):
        return np.argsort(scores)[::-1]
    idx = np.argpartition(scores, -top_k)[-top_k:]
    idx = idx[np.argsort(scores[idx])[::-1]]
    return idx


class TfidfRetriever:
    def __init__(self):
        self.vectorizer = None
        self.doc_vectors = None
        self.doc_ids = None

    def fit(self, docs: pd.DataFrame):
        self.vectorizer = TfidfVectorizer(
            lowercase=False,
            tokenizer=tokenize,
            ngram_range=CFG.tfidf_ngram_range,
            min_df=CFG.tfidf_min_df,
            sublinear_tf=CFG.tfidf_sublinear_tf,
        )
        self.doc_vectors = self.vectorizer.fit_transform(docs["content"])
        self.doc_ids = docs["id"].astype(str).tolist()
        return self

    def search(self, queries: pd.DataFrame, top_k: int) -> Dict[str, List[str]]:
        from sklearn.metrics.pairwise import cosine_similarity

        query_vectors = self.vectorizer.transform(queries["content"])
        pred_map: Dict[str, List[str]] = {}
        for i, (_, row) in enumerate(queries.iterrows()):
            scores = cosine_similarity(query_vectors[i], self.doc_vectors).ravel()
            idx = top_k_from_scores(scores, top_k)
            pred_map[str(row["id"])] = [self.doc_ids[j] for j in idx]
        return pred_map


class BM25PlusRetriever:
    def __init__(self):
        self.model = None
        self.doc_ids = None

    def fit(self, docs: pd.DataFrame):
        tokenized_docs = [tokenize(x) for x in docs["content"].tolist()]
        self.model = BM25Plus(
            tokenized_docs,
            k1=CFG.bm25_k1,
            b=CFG.bm25_b,
            delta=CFG.bm25_delta,
        )
        self.doc_ids = docs["id"].astype(str).tolist()
        return self

    def search(self, queries: pd.DataFrame, top_k: int) -> Dict[str, List[str]]:
        pred_map: Dict[str, List[str]] = {}
        for _, row in queries.iterrows():
            scores = np.asarray(self.model.get_scores(tokenize(row["content"])), dtype=float)
            idx = top_k_from_scores(scores, top_k)
            pred_map[str(row["id"])] = [self.doc_ids[j] for j in idx]
        return pred_map


class DenseRetriever:
    def __init__(self, model_name: str):
        self.model_name = model_name
        self.model = None
        self.doc_embeddings = None
        self.doc_ids = None

    def _load_model(self):
        if self.model is not None:
            return self.model
        try:
            from sentence_transformers import SentenceTransformer
        except ImportError:
            import sys, subprocess
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"])
            from sentence_transformers import SentenceTransformer
        self.model = SentenceTransformer(self.model_name)
        return self.model

    def _cache_path(self, docs: pd.DataFrame) -> Path:
        signature = dataframe_fingerprint(docs, ["id", "content"])
        return CACHE_DIR / f"dense_docs_{safe_name(self.model_name)}_{signature}.npy"

    def fit(self, docs: pd.DataFrame):
        self.doc_ids = docs["id"].astype(str).tolist()
        cache_path = self._cache_path(docs)
        cache_key = str(cache_path.resolve())

        if cache_key in ARRAY_CACHE:
            self.doc_embeddings = ARRAY_CACHE[cache_key]
            return self

        if CFG.enable_disk_cache and cache_path.exists():
            self.doc_embeddings = np.load(cache_path)
            ARRAY_CACHE[cache_key] = self.doc_embeddings
            return self

        model = self._load_model()
        self.doc_embeddings = model.encode(
            docs["content"].tolist(),
            batch_size=CFG.embedding_batch_size,
            show_progress_bar=True,
            normalize_embeddings=True,
            convert_to_numpy=True,
        )
        if CFG.enable_disk_cache:
            np.save(cache_path, self.doc_embeddings)
        ARRAY_CACHE[cache_key] = self.doc_embeddings
        return self

    def search(self, queries: pd.DataFrame, top_k: int) -> Dict[str, List[str]]:
        model = self._load_model()
        query_embeddings = model.encode(
            queries["content"].tolist(),
            batch_size=CFG.embedding_batch_size,
            show_progress_bar=True,
            normalize_embeddings=True,
            convert_to_numpy=True,
        )
        pred_map: Dict[str, List[str]] = {}
        for i, (_, row) in enumerate(queries.iterrows()):
            scores = self.doc_embeddings @ query_embeddings[i]
            idx = top_k_from_scores(scores, top_k)
            pred_map[str(row["id"])] = [self.doc_ids[j] for j in idx]
        return pred_map


In [ ]:

def rrf_fuse(rankings: Sequence[Dict[str, List[str]]], top_k: int, rrf_k: int = 60) -> Dict[str, List[str]]:
    fused: Dict[str, List[str]] = {}
    all_qids = set()
    for pred_map in rankings:
        all_qids.update(pred_map.keys())

    for qid in all_qids:
        scores = {}
        for pred_map in rankings:
            docs = pred_map.get(qid, [])
            for rank, doc_id in enumerate(docs, start=1):
                scores[doc_id] = scores.get(doc_id, 0.0) + 1.0 / (rrf_k + rank)
        ordered = sorted(scores.items(), key=lambda x: x[1], reverse=True)
        fused[qid] = [doc_id for doc_id, _ in ordered[:top_k]]
    return fused


def lexical_candidate_dense_rerank(
    tfidf_map: Dict[str, List[str]],
    bm25_map: Dict[str, List[str]],
    dense_map: Dict[str, List[str]],
    top_k: int,
) -> Dict[str, List[str]]:
    reranked = {}
    for qid in tfidf_map.keys():
        seen = set()
        candidates = []
        for source in (tfidf_map.get(qid, []), bm25_map.get(qid, []), dense_map.get(qid, [])):
            for doc_id in source:
                if doc_id not in seen:
                    seen.add(doc_id)
                    candidates.append(doc_id)
        reranked[qid] = candidates[:top_k]
    return reranked


def add_category_bonus(
    pred_map: Dict[str, List[str]],
    query_category_map: Dict[str, str],
    docs_frame: pd.DataFrame,
    bonus: float,
    top_k: int,
    rrf_k: int = 60,
) -> Dict[str, List[str]]:
    if "category" not in docs_frame.columns:
        return pred_map

    doc_category = docs_frame[["id", "category"]].copy()
    doc_category["id"] = doc_category["id"].astype(str)
    doc_to_cat = dict(zip(doc_category["id"], doc_category["category"].astype(str)))

    boosted = {}
    for qid, ranking in pred_map.items():
        target_cat = str(query_category_map.get(qid, ""))
        scores = {}
        for rank, doc_id in enumerate(ranking, start=1):
            score = 1.0 / (rrf_k + rank)
            if target_cat and str(doc_to_cat.get(str(doc_id), "")) == target_cat:
                score += bonus
            scores[str(doc_id)] = score
        ordered = sorted(scores.items(), key=lambda x: x[1], reverse=True)
        boosted[qid] = [doc_id for doc_id, _ in ordered[:top_k]]
    return boosted



## Experiment runner

This is the core evaluation block.

It performs the comparison requested by the competition and by your project brief:
- lexical baselines
- dense baselines
- fusion strategies
- category-aware reranking
- automatic `top_k` selection

The best configuration is selected from **held-out validation**, not from intuition.


In [ ]:

def run_validation_experiments(train_fold: pd.DataFrame, val_fold: pd.DataFrame, docs_df: pd.DataFrame) -> pd.DataFrame:
    experiments = []

    # Classifier metrics for Phase 2.
    clf_model = QueryCategoryClassifier().fit(train_fold)
    val_cat_pred = clf_model.predict(val_fold)
    class_metrics = evaluate_classification(val_fold["gold_category"].astype(str).tolist(), val_cat_pred)
    query_category_map = dict(zip(val_fold["id"].astype(str), val_cat_pred))

    max_k = max(CFG.top_k_grid)
    candidate_k = min(len(docs_df), max_k * CFG.embedding_candidate_multiplier)

    tfidf = TfidfRetriever().fit(docs_df)
    bm25 = BM25PlusRetriever().fit(docs_df)

    tfidf_full = tfidf.search(val_fold, candidate_k)
    bm25_full = bm25.search(val_fold, candidate_k)
    sparse_rrf_full = rrf_fuse([tfidf_full, bm25_full], top_k=candidate_k, rrf_k=CFG.rrf_k)

    base_systems: Dict[str, Dict[str, List[str]]] = {
        "tfidf": tfidf_full,
        "bm25plus": bm25_full,
        "tfidf+bm25plus_rrf": sparse_rrf_full,
    }

    for k in CFG.top_k_grid:
        for model_name, pred_map in base_systems.items():
            truncated = {qid: docs[:k] for qid, docs in pred_map.items()}
            metrics = evaluate_retrieval(truncated, val_fold, k)
            metrics.update(class_metrics if CFG.phase == 2 else {"accuracy": 0.0})
            experiments.append({
                "model": model_name,
                "dense_model": None,
                "uses_category_bonus": False,
                "k": k,
                **metrics,
                "score": combined_score(metrics, k, CFG.phase),
            })

    for dense_model_name in CFG.embedding_model_names:
        dense = DenseRetriever(dense_model_name).fit(docs_df)
        dense_full = dense.search(val_fold, candidate_k)
        hybrid_rrf_full = rrf_fuse([tfidf_full, bm25_full, dense_full], top_k=candidate_k, rrf_k=CFG.rrf_k)
        dense_union_full = lexical_candidate_dense_rerank(tfidf_full, bm25_full, dense_full, top_k=candidate_k)

        dense_systems: Dict[str, Dict[str, List[str]]] = {
            "dense": dense_full,
            "tfidf+bm25plus+dense_rrf": hybrid_rrf_full,
            "candidate_union": dense_union_full,
        }

        for k in CFG.top_k_grid:
            for system_name, pred_map in dense_systems.items():
                truncated = {qid: docs[:k] for qid, docs in pred_map.items()}
                metrics = evaluate_retrieval(truncated, val_fold, k)
                metrics.update(class_metrics if CFG.phase == 2 else {"accuracy": 0.0})
                experiments.append({
                    "model": system_name,
                    "dense_model": dense_model_name,
                    "uses_category_bonus": False,
                    "k": k,
                    **metrics,
                    "score": combined_score(metrics, k, CFG.phase),
                })

                if CFG.phase == 2 and CFG.enable_category_bonus_experiments:
                    boosted = add_category_bonus(
                        pred_map=truncated,
                        query_category_map=query_category_map,
                        docs_frame=docs_df,
                        bonus=CFG.category_bonus,
                        top_k=k,
                        rrf_k=CFG.rrf_k,
                    )
                    boosted_metrics = evaluate_retrieval(boosted, val_fold, k)
                    boosted_metrics.update(class_metrics)
                    experiments.append({
                        "model": system_name,
                        "dense_model": dense_model_name,
                        "uses_category_bonus": True,
                        "k": k,
                        **boosted_metrics,
                        "score": combined_score(boosted_metrics, k, CFG.phase),
                    })

    leaderboard_df = pd.DataFrame(experiments).sort_values("score", ascending=False).reset_index(drop=True)
    return leaderboard_df


In [ ]:

leaderboard_df = run_validation_experiments(train_fold, val_fold, docs_df)
display(leaderboard_df.head(20))

best_config = leaderboard_df.iloc[0].to_dict()
print("Best validation configuration:")
print(best_config)



## Final training and test-set prediction

After selecting the best validation configuration, the notebook retrains the classifier on **all training queries** and builds the final retrieval system for the test queries.

This ensures the final submission uses the full training signal while still respecting validation during model selection.


In [ ]:

def train_final_system_and_predict(
    train_df: pd.DataFrame,
    queries_test_df: pd.DataFrame,
    docs_df: pd.DataFrame,
    best_config: Dict[str, Any],
):
    chosen_k = int(best_config["k"])
    chosen_model = str(best_config["model"])
    chosen_dense_model = best_config.get("dense_model")
    use_category_bonus = bool(best_config.get("uses_category_bonus", False))

    final_clf = QueryCategoryClassifier().fit(train_df)
    test_category_preds = final_clf.predict(queries_test_df)
    test_category_map = dict(zip(queries_test_df["id"].astype(str), test_category_preds))

    candidate_k = min(len(docs_df), chosen_k * CFG.embedding_candidate_multiplier)

    tfidf = TfidfRetriever().fit(docs_df)
    bm25 = BM25PlusRetriever().fit(docs_df)

    tfidf_pred = tfidf.search(queries_test_df, candidate_k)
    bm25_pred = bm25.search(queries_test_df, candidate_k)

    if chosen_model == "tfidf":
        final_pred = {qid: docs[:chosen_k] for qid, docs in tfidf_pred.items()}
    elif chosen_model == "bm25plus":
        final_pred = {qid: docs[:chosen_k] for qid, docs in bm25_pred.items()}
    elif chosen_model == "tfidf+bm25plus_rrf":
        final_pred = rrf_fuse([tfidf_pred, bm25_pred], top_k=chosen_k, rrf_k=CFG.rrf_k)
    else:
        dense_model_name = str(chosen_dense_model) if chosen_dense_model else CFG.embedding_model_names[0]
        dense = DenseRetriever(dense_model_name).fit(docs_df)
        dense_pred = dense.search(queries_test_df, candidate_k)

        if chosen_model == "dense":
            final_pred = {qid: docs[:chosen_k] for qid, docs in dense_pred.items()}
        elif chosen_model == "candidate_union":
            final_pred = lexical_candidate_dense_rerank(tfidf_pred, bm25_pred, dense_pred, top_k=chosen_k)
        else:
            final_pred = rrf_fuse([tfidf_pred, bm25_pred, dense_pred], top_k=chosen_k, rrf_k=CFG.rrf_k)

    if CFG.phase == 2 and use_category_bonus:
        final_pred = add_category_bonus(
            pred_map=final_pred,
            query_category_map=test_category_map,
            docs_frame=docs_df,
            bonus=CFG.category_bonus,
            top_k=chosen_k,
            rrf_k=CFG.rrf_k,
        )

    return final_pred, test_category_preds


final_pred_map, final_test_categories = train_final_system_and_predict(
    train_df=queries_train_df,
    queries_test_df=queries_test_df,
    docs_df=docs_df,
    best_config=best_config,
)

print("Predictions generated for:", len(final_pred_map), "queries")


In [ ]:

def write_submission(
    queries_test_df: pd.DataFrame,
    pred_map: Dict[str, List[str]],
    category_preds: Optional[List[str]],
    output_path: Path,
    phase: int,
) -> pd.DataFrame:
    rows = []
    qids = queries_test_df["id"].astype(str).tolist()
    for i, qid in enumerate(qids):
        doc_ids = [str(x) for x in pred_map[qid]]
        category_value = "" if phase == 1 else str(category_preds[i])
        rows.append({
            "query_id": qid,
            "relevant_doc_ids": json.dumps(doc_ids),
            "category": category_value,
        })

    submission_df = pd.DataFrame(rows, columns=["query_id", "relevant_doc_ids", "category"])
    submission_df.to_csv(output_path, index=False)
    return submission_df


submission_df = write_submission(
    queries_test_df=queries_test_df,
    pred_map=final_pred_map,
    category_preds=final_test_categories if CFG.phase == 2 else None,
    output_path=OUTPUT_PATH,
    phase=CFG.phase,
)

display(submission_df.head())
print("Saved submission to:", OUTPUT_PATH.resolve())



## Report section — what was improved and why

This final section is the written project report requested for the notebook itself.

### 1. Baseline correction and cleanup
The notebook was reorganized around a single reproducible workflow instead of mixing exploratory code, submission-only code, and evaluation logic in a way that makes model comparison hard. The revised notebook keeps the cleaner, compact structure associated with the Maksym version, then adds the missing experimental rigor.

### 2. Proper offline validation
A major improvement was to evaluate on a **held-out validation split** instead of tuning directly on all training queries. This matters because retrieval systems can appear much stronger than they really are if `top_k`, fusion style, or classification decisions are chosen on the same data used for measurement.

### 3. Required model comparison
The notebook now compares all retrieval families required by the competition brief:
- TF-IDF
- BM25+
- Embedding retrieval
- Hybrid cooperation methods

This makes the final selection evidence-driven rather than based on one preferred method.

### 4. BM25+ instead of weaker BM25 baselines
The lexical component uses **BM25+**, which is usually a better choice than simpler BM25 variants when document-length effects and score floor behavior matter.

### 5. Classification integrated into retrieval optimization
For Phase 2, the notebook trains a query-domain classifier and uses it for **category-aware reranking**. This is important because the leaderboard score includes classification accuracy and because correct domain signals can improve retrieval ranking quality at the same time.

### 6. Cooperation strategies instead of isolated models
A good IR system is often not a single model but a cooperation pipeline. This notebook therefore evaluates:
- sparse-only retrieval
- dense-only retrieval
- reciprocal-rank fusion
- category-guided reranking

This allows the final system to exploit both lexical exact matching and semantic similarity.

### 7. Automatic `top_k` selection
The competition explicitly treats retrieval depth as part of system design. The notebook therefore sweeps a configurable `top_k` grid and selects the best value according to the weighted offline score.

### 8. Caching for faster iteration
Dense retrieval can be slow because document embeddings must be computed over the entire corpus. To make iteration practical, the notebook caches expensive artifacts such as document embeddings. This means repeated model selection runs are much faster.

### 9. Submission correctness
The final submission writer follows the competition format exactly:
- `query_id`
- `relevant_doc_ids` as a JSON list string
- `category`

For **Phase 1**, the category column is written as an empty string. For **Phase 2**, the predicted class label is written.

### 10. Final decision rule
The final system is not hardcoded in advance. Instead, the notebook:
1. trains models on the training fold,
2. evaluates on the validation fold,
3. ranks configurations by the weighted score,
4. retrains the best configuration on all training queries,
5. generates the final Kaggle submission.

### 11. What should be reported after running the notebook
After execution, the team report should cite:
- the best validation configuration,
- the best `top_k`,
- the best retrieval family,
- whether category-aware reranking helped,
- the classifier validation accuracy,
- the final reasoning for the selected submission.

### 12. Practical conclusion
The main change in this revised notebook is not only the addition of more models. The critical improvement is the shift from a notebook that can produce a submission to a notebook that can **justify** why a specific submission should be trusted.



## Short usage notes

- Run the notebook once with the default configuration.
- Inspect `leaderboard_df` and the printed `best_config`.
- If compute budget allows, expand `CFG.top_k_grid` and optionally add more embedding models.
- Submit `solutions_SeaFour.csv`.

For stronger final reporting, copy the actual validation numbers from `leaderboard_df.head(20)` into your written project document or Kaggle notes.
